# HealthConnect Rwanda — Clinical Risk Prediction & Healthcare Intelligence
### Databricks edition (final consolidated pipeline)

One notebook that takes the 28 raw files to **Delta tables, MLflow-tracked models, anomaly/segment/capacity outputs, explainability, a gated model-promotion decision and a daily retrain-and-score job**.

**Run guide**
1. **Cluster:** Databricks Runtime 14.3 LTS ML or newer (single node is enough; ~8 GB RAM). Unity Catalog recommended.
2. **Upload the raw files** to a Unity Catalog Volume (e.g. `/Volumes/main/healthconnect/raw`) using the original file names (see the check in section 1). `HC_Patients_1.csv`, Rooms/Services/Equipment CSVs are duplicates and not needed.
3. **Set the widgets** at the top of the notebook: `raw_dir`, `out_dir`, `daily_dir`, `catalog_schema` (leave blank to write CSVs instead of Delta), `mlflow_experiment`, `register_models`.
4. **Run all.** Expect roughly 10 to 20 minutes on a small cluster (most of the time is cross-validation of 5 algorithms × 7 tasks).
5. **Schedule** the notebook as a daily Workflow (section 13) to ingest the day's CSV drop, check drift, retrain and score.

| Stage | Section | Output |
|---|---|---|
| Config, widgets, helpers | 0 | paths, MLflow experiment, Delta schema |
| Load and validate | 1 | 20 required files, coverage table |
| Clean, integrate, features | 2 | appointment table (5,000 rows), billing table (60,000 rows) |
| Models | 3 | 7 tasks × 5 algorithms, MLflow runs, optional UC registry |
| Segments, capacity | 4 | patient/facility segments, staffing pressure |
| Anomalies | 5 | review lists |
| Explainability | 6 | SHAP, permutation, LIME, bootstrap CI |
| Compare, decide | 7 | model table, deployment verdict |
| Gold tables | 10 | Delta tables for AI/BI dashboards |
| Daily operations | 11 | validate → drift → retrain → gate → score |

> **Data caveats that drive every result:** the Appointments sheet holds only 5,000 of 60,000 appointments; the claims table has only Approved/Pending statuses; billing status contradicts payment amounts; facilities have no location; and the data show almost no predictive signal, so metrics are always compared with naive baselines and no model is promoted below the agreed bar (CV AUC ≥ 0.70, R² > 0.05).

In [ ]:
%pip install -q shap lime xgboost lightgbm openpyxl mlflow

In [ ]:
# Restart Python so the new packages are importable (no-op outside Databricks)
try:
    dbutils.library.restartPython()
except NameError:
    pass

## 0 · Configuration, widgets and helpers

In [ ]:
import warnings, re, json, os, glob, html
os.environ.setdefault('MLFLOW_ALLOW_FILE_STORE', 'true')   # only matters for local file-store tests; Databricks uses its managed tracking server
from pathlib import Path
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from IPython.display import Markdown, display as _ipy_display
from sklearn.base import clone
from sklearn.model_selection import train_test_split, StratifiedKFold, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, GradientBoostingRegressor, IsolationForest
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay,
                             mean_squared_error, mean_absolute_error, r2_score, silhouette_score)
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.inspection import permutation_importance
from xgboost import XGBClassifier, XGBRegressor
from lightgbm import LGBMClassifier
import mlflow, mlflow.sklearn, logging
logging.getLogger('mlflow').setLevel(logging.ERROR)   # hide routine MLflow pickle/serialization notices
from mlflow.models import infer_signature
from mlflow.tracking import MlflowClient

IS_DBX = 'DATABRICKS_RUNTIME_VERSION' in os.environ

# ---- display(): Databricks table/HTML rendering on the cluster, plain IPython elsewhere ----
try: _dbx_display = display            # injected by Databricks
except NameError: _dbx_display = None
def _md2html(t):
    t = html.escape(t); t = re.sub(r'\*\*(.+?)\*\*', r'<b>\1</b>', t); t = re.sub(r'(?<!\*)\*(.+?)\*(?!\*)', r'<i>\1</i>', t)
    return t.replace('\n', '<br>')
def display(obj):
    if isinstance(obj, Markdown):
        displayHTML(f'<div style="font-family:sans-serif;font-size:14px">{_md2html(obj.data)}</div>') if IS_DBX else _ipy_display(obj)
    elif isinstance(obj, pd.DataFrame) and IS_DBX and _dbx_display:
        d = obj if isinstance(obj.index, pd.RangeIndex) else obj.reset_index(); d = d.copy(); d.columns = [str(c) for c in d.columns]
        for c in d.columns[d.dtypes.astype(str).isin(['object', 'category'])]: d[c] = d[c].astype(str)
        _dbx_display(d)
    else: _ipy_display(obj)

# ---- widgets (Databricks) or environment variables / defaults (local test) ----
def W(name, default, label=None):
    if IS_DBX:
        try: dbutils.widgets.text(name, default, label or name)
        except Exception: pass
        return dbutils.widgets.get(name).strip()
    return os.environ.get('HC_' + name.upper(), default).strip()
RAW_DIR   = W('raw_dir', '/Volumes/main/healthconnect/raw' if IS_DBX else '/mnt/user-data/uploads', 'Raw files folder')
OUT_DIR   = W('out_dir', '/Volumes/main/healthconnect/outputs' if IS_DBX else '/home/claude/dbx_out', 'Outputs folder')
DAILY_DIR = W('daily_dir', '/Volumes/main/healthconnect/daily' if IS_DBX else '/tmp/daily', 'Daily CSV drop folder')
CATALOG   = W('catalog_schema', 'main.healthconnect' if IS_DBX else '', 'Delta catalog.schema (blank = CSV)')
EXPERIMENT = W('mlflow_experiment', '/Shared/healthconnect-ml' if IS_DBX else 'healthconnect-ml', 'MLflow experiment')
REGISTER  = W('register_models', 'false', 'Register models in UC (true/false)').lower() == 'true'

CFG = dict(IN=RAW_DIR.rstrip('/') + '/', OUT=OUT_DIR.rstrip('/') + '/', SEED=42, DEPLOY_AUC=0.70, DEPLOY_R2=0.05)
RS = CFG['SEED']; np.random.seed(RS)
for p in [CFG['OUT'] + 'figures', CFG['OUT'] + 'tables']: os.makedirs(p, exist_ok=True)
sns.set_theme(style='whitegrid'); plt.rcParams.update({'figure.dpi': 100, 'axes.spines.top': False, 'axes.spines.right': False})
clean_cols = lambda cols: [re.sub(r'[^0-9a-zA-Z_]+', '_', c.replace('+', 'pos').replace('-', 'neg')) for c in cols]
LOG = []
def log(step, finding, action): LOG.append(dict(Step=step, Finding=finding, Action=action))

# ---- MLflow + Delta ----
if not IS_DBX: mlflow.set_tracking_uri('file:' + os.environ.get('HC_MLRUNS', '/home/claude/mlruns'))
mlflow.set_experiment(EXPERIMENT)
if REGISTER and CATALOG: mlflow.set_registry_uri('databricks-uc')
if IS_DBX and CATALOG:
    try: spark.sql(f'CREATE SCHEMA IF NOT EXISTS {CATALOG}')
    except Exception as e: print('Could not create schema (check permissions):', e)
TABLES = {}
def write_table(df, name):
    """Delta table `<catalog.schema>.<name>` on Databricks when catalog_schema is set; otherwise a CSV under <out_dir>/tables."""
    d = df.copy(); d.columns = [re.sub(r'[^0-9a-zA-Z_]+', '_', str(c)).strip('_') for c in d.columns]
    for c in d.columns[d.dtypes.astype(str).isin(['object', 'category'])]: d[c] = d[c].astype(str)
    if IS_DBX and CATALOG:
        spark.createDataFrame(d).write.mode('overwrite').option('overwriteSchema', 'true').saveAsTable(f'{CATALOG}.{name}'); ref = f'delta:{CATALOG}.{name}'
    else:
        p = Path(CFG['OUT']) / 'tables'; p.mkdir(parents=True, exist_ok=True); d.to_csv(p / f'{name}.csv', index=False); ref = str(p / f'{name}.csv')
    TABLES[name] = ref; return ref
print(f'Databricks: {IS_DBX} | raw: {RAW_DIR} | out: {OUT_DIR} | delta: {CATALOG or "off (CSV)"} | experiment: {EXPERIMENT} | register: {REGISTER}')

## 1 · Data loading and integration map
```
Country → Province → District → Sector → Village → Patients ─┐
Departments(Facility) → Rooms → Beds                          ├─< Appointments >─ Doctors / Departments / Facilities
Staff: doctors, nurses, admin, pharmacists, lab scientists, radiographers (Facility, Department)
Appointments ─1:1─ Diagnoses · Billing ─1:1─ Payments ;  Billing ─0..1─ Insurance_Claims
Appointments ─0..1─ Laboratory · Prescriptions · Admissions · Referrals · Feedback ;  Revenue_Summary (monthly totals)
```

In [ ]:
CSV_FILES = dict(patients='HC_Patients', billing='HC_Billing', payments='HC_Payments', claims='Insurance_Claims', diagnoses='HC_Diagnoses', feedback='HC_Patient_Feedback',
                 doctors='HC_Doctor', nurses='HC_Nurse', admin='HC_Administrative_Staff', pharm='Pharmacist', labsci='Laboratory_Scientist', rad='HC_Radiographer',
                 dept='HC_Departments', villages='HC_Villages', sectors='HC_Sectors', districts='HC_Districts', provinces='HC_Provinces', revenue='HC_Revenue_Summary')
XLSX_FILES = ['HC_Clinical_Transactions', 'HC_Facility_Master_Data']
URI = ('s3:', 'abfss:', 'gs:', 'wasbs:')
_local = lambda p: p.replace('dbfs:/', '/dbfs/') if p.startswith('dbfs:/') else p
def read_csv_any(name):
    p = f'{RAW_DIR.rstrip("/")}/{name}.csv'
    if p.startswith(URI): return spark.read.option('header', True).option('inferSchema', True).csv(p).toPandas()   # cloud URIs via Spark
    return pd.read_csv(_local(p))
def read_xlsx(name): return pd.read_excel(_local(f'{RAW_DIR.rstrip("/")}/{name}.xlsx'), sheet_name=None)

def check_inputs():
    if RAW_DIR.startswith(URI): return
    need = [f'{n}.csv' for n in CSV_FILES.values()] + [f'{n}.xlsx' for n in XLSX_FILES]
    missing = [f for f in need if not os.path.exists(_local(f'{RAW_DIR.rstrip("/")}/{f}'))]
    if missing: raise FileNotFoundError(f'{len(missing)} required file(s) missing from {RAW_DIR}: {missing}. Upload them (original names) or change the raw_dir widget.')
    print(f'All {len(need)} required input files found in {RAW_DIR}')

def load_data():
    """Load every supplied file into one dict of DataFrames."""
    ct, fm = read_xlsx('HC_Clinical_Transactions'), read_xlsx('HC_Facility_Master_Data')
    T = dict(appointments=ct['Appointments'], labs=ct['Laboratory_Tests'], rx=ct['Prescriptions'], admissions=ct['Admissions'], referrals=ct['Referrals'], rooms=fm['Rooms'], beds=fm['Beds'])
    T.update({k: read_csv_any(v) for k, v in CSV_FILES.items()})
    T['staff'] = pd.concat([T[k].assign(Role=T[k].get('Role', k)) for k in ['doctors', 'nurses', 'admin', 'pharm', 'labsci', 'rad']], ignore_index=True)
    return T
check_inputs(); T = load_data()
cov = pd.DataFrame({t: dict(rows=len(T[t]), linkable_to_appointments=int(T[t].AppointmentID.isin(T['appointments'].AppointmentID).sum()))
                    for t in ['diagnoses', 'billing', 'feedback', 'labs', 'rx', 'admissions', 'referrals']}).T
display(cov); print('Appointments sheet rows:', len(T['appointments']), '| tables loaded:', len(T))
assert len(T['appointments']) > 0 and T['billing'].BillingID.is_unique, 'Input validation failed'

## 2 · Cleaning, integration and feature engineering
Two modelling tables are built: the **appointment table** (one row per appointment; patient history uses *earlier visits only* to avoid leakage) and the **billing table** (all 60,000 bills + payment + claim + diagnosis + rating).

In [ ]:
def clean_patients(T):
    p = T['patients'].copy()
    p['DateOfBirth'] = pd.to_datetime(p.DateOfBirth, format='%m/%d/%Y', errors='coerce')
    age = ((pd.Timestamp('2026-12-31') - p.DateOfBirth).dt.days // 365.25); bad = (p.Age - age).abs() > 1; p.loc[bad, 'Age'] = age[bad]
    log('Age vs DOB', f'{bad.sum()} mismatches', 'recomputed')
    log('Missing ChronicCondition', f'{p.ChronicCondition.isna().mean():.0%}', "imputed 'None'"); p['ChronicCondition'] = p.ChronicCondition.fillna('None')
    p['BMI_flag'] = ((p.BMI > 45) | (p.BMI < 12) | ((p.Age < 18) & (p.BMI > 35))).astype(int); lo, hi = p.BMI.quantile([.01, .99]); p['BMI_clean'] = p.BMI.clip(lo, hi)
    log('Implausible BMI', f'{p.BMI_flag.mean():.1%}', 'flagged + winsorised')
    log('Phone numbers', f'{p.Phone.nunique()} distinct values / {len(p):,} patients (digits lost)', 'unusable for identity matching')
    log('Date of birth', f'{(p.DateOfBirth.dt.month.eq(1) & p.DateOfBirth.dt.day.eq(1)).mean():.0%} fall on 1 January', 'treated as year-only')
    geo = (T['villages'].merge(T['sectors'], on='SectorID').merge(T['districts'], on='DistrictID').merge(T['provinces'], on='ProvinceID'))[['VillageID', 'DistrictID', 'ProvinceName']]
    return p.merge(geo, on='VillageID', how='left')

def build_billing_table(T):
    """Billing grain: bill + payment + claim + diagnosis + rating (60,000 rows)."""
    b = (T['billing'].rename(columns={'Status': 'BillStatus'}).merge(T['payments'].rename(columns={'Method': 'PayMethod'}), on='BillingID')
         .merge(T['claims'].rename(columns={'Status': 'ClaimStatus'}), on='BillingID', how='left')
         .merge(T['diagnoses'][['AppointmentID', 'Diagnosis', 'Severity']], on='AppointmentID').merge(T['feedback'][['AppointmentID', 'Rating']], on='AppointmentID', how='left'))
    b['pay_ratio'] = b.AmountPaid / b.Amount; b['HasClaim'] = b.ClaimStatus.notna().astype(int); b['claim_ratio'] = (b.ClaimAmount / b.Amount).fillna(0)
    b['Provider'] = b.Provider.fillna('NoClaim'); b['ClaimStatus'] = b.ClaimStatus.fillna('NoClaim')
    b['SatisfactionClass'] = pd.cut(b.Rating, [0, 2, 3, 5], labels=['Low', 'Medium', 'High'])
    log('Billing status vs payment', f"{((b.BillStatus=='Pending') & (b.pay_ratio>=1)).sum():,} 'Pending' bills fully paid; {((b.BillStatus=='Paid') & (b.pay_ratio<1)).sum():,} 'Paid' under-paid", 'status judged unreliable; payment ratio used')
    log('Claims', f"{b.HasClaim.sum():,} claims, statuses {T['claims'].Status.value_counts().to_dict()}", 'binary Approved/Pending model (no Rejected exists)')
    return b

def build_appointment_table(T, patients):
    """Appointment grain with targets, history, workload and calendar features."""
    a = T['appointments'].copy(); a['Date'] = pd.to_datetime(a.Date)
    dept_fac = dict(zip(T['dept'].DepartmentID, T['dept'].FacilityID)); a['dept_fac_ok'] = (a.DepartmentID.map(dept_fac) == a.FacilityID).astype(int)
    log('Department belongs to appointment facility', f'{1 - a.dept_fac_ok.mean():.1%} inconsistent', 'flagged; department→facility map used for capacity analysis')
    doc = T['doctors'][['StaffID', 'Specialty', 'YearsExperience']].rename(columns={'StaffID': 'DoctorID', 'Specialty': 'DoctorSpecialty', 'YearsExperience': 'DoctorExperience'})
    m = (a.merge(patients, on='PatientID', how='left').merge(T['diagnoses'][['AppointmentID', 'Diagnosis', 'Severity']], on='AppointmentID', how='left')
          .merge(T['billing'][['AppointmentID', 'BillingID', 'Amount', 'Status']].rename(columns={'Status': 'BillStatus'}), on='AppointmentID', how='left')
          .merge(T['feedback'][['AppointmentID', 'Rating']], on='AppointmentID', how='left').merge(T['labs'][['AppointmentID', 'Result']].rename(columns={'Result': 'LabResult'}), on='AppointmentID', how='left')
          .merge(T['rx'][['AppointmentID', 'Medication']], on='AppointmentID', how='left').merge(T['admissions'][['AppointmentID', 'LengthOfStayDays']], on='AppointmentID', how='left')
          .merge(T['referrals'][['AppointmentID', 'ReferredFacilityID']], on='AppointmentID', how='left').merge(doc, on='DoctorID', how='left')
          .merge(T['dept'][['DepartmentID', 'DepartmentName']], on='DepartmentID', how='left')).sort_values(['PatientID', 'Date', 'AppointmentID']).reset_index(drop=True)
    # targets
    m['Admitted'] = m.LengthOfStayDays.notna().astype(int); m['Referred'] = m.ReferredFacilityID.notna().astype(int); m['HighRisk'] = ((m.Admitted == 1) | (m.Referred == 1)).astype(int)
    # history (earlier visits only)
    g = m.groupby('PatientID', sort=False); m['has_lab'] = m.LabResult.notna().astype(int); m['lab_abn'] = (m.LabResult == 'Abnormal').astype(int); m['has_rx'] = m.Medication.notna().astype(int); m['severe'] = (m.Severity == 'Severe').astype(int)
    m['n_prev_appointments'] = g.cumcount()
    for col, new in [('Admitted', 'n_prev_admissions'), ('Referred', 'n_prev_referrals'), ('has_lab', 'n_prev_labs'), ('lab_abn', 'n_prev_abnormal_labs'), ('has_rx', 'n_prev_prescriptions'), ('severe', 'n_prev_severe')]:
        m[new] = g[col].cumsum() - m[col]
    r = m.Rating.fillna(0); h = m.Rating.notna().astype(int); cn = h.groupby(m.PatientID).cumsum() - h
    m['prev_satisfaction'] = np.where(cn > 0, (r.groupby(m.PatientID).cumsum() - r) / cn.replace(0, np.nan), np.nan); m['prev_satisfaction'] = m.prev_satisfaction.fillna(m.Rating.mean())
    m['days_since_last_visit'] = g['Date'].diff().dt.days.fillna(-1)
    m['n_chronic_conditions'] = (m.ChronicCondition != 'None').astype(int) + (m.groupby('PatientID').Diagnosis.transform(lambda s: s.isin(['Diabetes', 'Hypertension']).cumsum().shift(1).fillna(0)) > 0).astype(int)
    # demographic / calendar / workload
    m['AgeGroup'] = pd.cut(m.Age, [-1, 4, 14, 24, 44, 64, 120], labels=['0-4', '5-14', '15-24', '25-44', '45-64', '65+']).astype(str)
    m['Month'] = m.Date.dt.month; m['DayOfWeek'] = m.Date.dt.dayofweek; m['Quarter'] = m.Date.dt.quarter; m['YM'] = m.Date.dt.to_period('M').astype(str)
    m['Season'] = m.Month.map(lambda x: 'ShortDry' if x < 3 else 'LongRains' if x < 6 else 'LongDry' if x < 9 else 'ShortRains')
    m['doctor_workload'] = m.groupby(['DoctorID', 'YM']).AppointmentID.transform('count'); m['department_workload'] = m.groupby(['DepartmentID', 'YM']).AppointmentID.transform('count')
    m['facility_monthly_volume'] = m.groupby(['FacilityID', 'YM']).AppointmentID.transform('count'); m['facility_daily_load'] = m.groupby(['FacilityID', 'Date']).AppointmentID.transform('count')   # waiting-time proxy
    beds_f = T['beds'].merge(T['rooms'][['RoomID', 'FacilityID']], on='RoomID').groupby('FacilityID').size().rename('facility_beds'); m = m.merge(beds_f, on='FacilityID', how='left')
    m['facility_utilization'] = m.facility_monthly_volume / m.facility_beds; m['network_monthly_appts'] = m.YM.map(T['revenue'].set_index('Month').Appointments)
    m['RevenueCategory'] = pd.qcut(m.Amount, 3, labels=['Low', 'Medium', 'High']).astype(str)
    return m

patients = clean_patients(T); bf = build_billing_table(T); m = build_appointment_table(T, patients)
CAT = ['Gender', 'MaritalStatus', 'Education', 'BloodGroup', 'ChronicCondition', 'AgeGroup', 'VisitType', 'Diagnosis', 'Severity', 'Season', 'DepartmentName', 'FacilityID', 'InsuranceID', 'DoctorSpecialty', 'ProvinceName', 'DistrictID']
NUM = ['Age', 'BMI_clean', 'BMI_flag', 'n_prev_appointments', 'n_prev_admissions', 'n_prev_referrals', 'n_prev_labs', 'n_prev_abnormal_labs', 'n_prev_prescriptions', 'n_prev_severe', 'n_chronic_conditions',
       'prev_satisfaction', 'days_since_last_visit', 'doctor_workload', 'department_workload', 'facility_monthly_volume', 'facility_daily_load', 'facility_utilization', 'DoctorExperience', 'Month', 'Quarter', 'DayOfWeek', 'network_monthly_appts']
def make_X(df, cat, num): return pd.concat([df[num].astype(float), pd.get_dummies(df[cat].astype(str), dtype=float)], axis=1).pipe(lambda x: x.set_axis(clean_cols(x.columns), axis=1))
X_app = make_X(m, CAT, NUM)
print(f'Appointment table {m.shape} | billing table {bf.shape} | model matrix {X_app.shape}')
print(f'Admitted {m.Admitted.mean():.1%} · Referred {m.Referred.mean():.1%} · High-risk {m.HighRisk.mean():.1%} · Claims approved {(bf.ClaimStatus=="Approved").sum()/bf.HasClaim.sum():.1%}')
display(pd.DataFrame(LOG))
# save cleaned datasets (direct identifiers removed)
m.drop(columns=['FirstName', 'LastName', 'Phone', 'EmergencyContact']).to_csv(CFG['OUT'] + 'appointment_table_clean.csv', index=False); bf.to_csv(CFG['OUT'] + 'billing_table_clean.csv', index=False)
pd.DataFrame(LOG).to_csv(CFG['OUT'] + 'preprocessing_log.csv', index=False)

In [ ]:
# ---- silver layer: cleaned, de-identified tables ----
IDENT = ['FirstName', 'LastName', 'Phone', 'EmergencyContact']
print(write_table(m.drop(columns=[c for c in IDENT if c in m]), 'hc_appointment_clean'))
print(write_table(bf, 'hc_billing_clean'))
print(write_table(pd.DataFrame(LOG), 'hc_preprocessing_log'))

**Engineered features and why:** age group (risk is non-linear in age) · previous appointments / admissions / referrals / labs / prescriptions / severe diagnoses (past escalation predicts future escalation) · chronic-condition count · doctor, department and facility workload and `facility_utilization` (crowding) · `facility_daily_load` (waiting-time proxy; no timestamps exist) · `prev_satisfaction` · season and network monthly load (seasonal disease patterns) · province/district (residence; true distance needs facility coordinates, which are not supplied) · `RevenueCategory` (finance segmentation; excluded from revenue model).
## 3 · Supervised models (classification and regression)
A **task registry** drives all models. Every task uses a stratified 80/20 split plus cross-validation, five algorithms, class weights for imbalance, and a naive baseline (majority class / mean) so that "good-looking" accuracy is never mistaken for skill.

In [ ]:
CLS = lambda multi=False: {
    'Logistic Regression': make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000, class_weight='balanced', C=0.1)),
    'Decision Tree': DecisionTreeClassifier(max_depth=5, min_samples_leaf=25, class_weight='balanced', random_state=RS),
    'Random Forest': RandomForestClassifier(n_estimators=200, max_depth=7, min_samples_leaf=15, class_weight='balanced', n_jobs=-1, random_state=RS),
    'XGBoost': XGBClassifier(n_estimators=150, max_depth=3, learning_rate=.05, subsample=.8, colsample_bytree=.7, eval_metric='mlogloss' if multi else 'logloss', random_state=RS, verbosity=0, n_jobs=4),
    'LightGBM': LGBMClassifier(n_estimators=150, num_leaves=12, learning_rate=.05, class_weight='balanced', random_state=RS, verbose=-1, n_jobs=4)}
REG = lambda: {'Linear Regression': make_pipeline(StandardScaler(), LinearRegression()), 'Random Forest': RandomForestRegressor(n_estimators=150, min_samples_leaf=20, max_depth=8, n_jobs=-1, random_state=RS),
               'XGBoost': XGBRegressor(n_estimators=150, max_depth=3, learning_rate=.05, subsample=.8, random_state=RS, verbosity=0, n_jobs=4), 'Gradient Boosting': GradientBoostingRegressor(n_estimators=120, max_depth=3, learning_rate=.05, subsample=.8, random_state=RS)}
def vg(auc): return 'no signal' if auc < .55 else 'weak' if auc < .65 else 'moderate' if auc < .75 else 'good' if auc < .85 else 'strong'

def run_task(name, X, y, kind):
    """kind: 'binary' | 'multi' | 'reg'. Returns result rows; stores fitted models/split in STORE."""
    y = np.asarray(y); strat = y if kind != 'reg' else None; Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=.2, stratify=strat, random_state=RS); rows = []; fitted = {}
    folds = 5 if len(X) < 10000 else 3; cvsplit = (StratifiedKFold if kind != 'reg' else KFold)(folds, shuffle=True, random_state=RS)
    models = REG() if kind == 'reg' else CLS(kind == 'multi')
    for algo, mdl in models.items():
        cv = cross_val_score(mdl, Xtr, ytr, cv=cvsplit, scoring={'binary': 'roc_auc', 'multi': 'f1_macro', 'reg': 'r2'}[kind])
        if kind == 'binary' and algo == 'XGBoost': mdl.set_params(scale_pos_weight=(ytr == 0).sum() / max((ytr == 1).sum(), 1))
        mdl.fit(Xtr, ytr); fitted[algo] = mdl
        if kind == 'reg':
            p = mdl.predict(Xte); rows.append(dict(Task=name, Kind=kind, Algorithm=algo, Primary='R²', Score=r2_score(yte, p), CV=cv.mean(), RMSE=np.sqrt(mean_squared_error(yte, p)), MAE=mean_absolute_error(yte, p), MAPE=np.mean(np.abs((yte - p) / yte)) * 100,
                                                  Baseline=0.0))
        else:
            pr_ = mdl.predict_proba(Xte); pred = pr_.argmax(1) if kind == 'multi' else (pr_[:, 1] >= .5).astype(int); avg = 'macro' if kind == 'multi' else 'binary'
            auc = roc_auc_score(yte, pr_, multi_class='ovr') if kind == 'multi' else roc_auc_score(yte, pr_[:, 1])
            rows.append(dict(Task=name, Kind=kind, Algorithm=algo, Primary='Macro-F1' if kind == 'multi' else 'ROC-AUC', Score=f1_score(yte, pred, average='macro') if kind == 'multi' else auc, CV=cv.mean(), AUC=auc,
                             Accuracy=accuracy_score(yte, pred), Precision=precision_score(yte, pred, average=avg, zero_division=0), Recall=recall_score(yte, pred, average=avg), F1=f1_score(yte, pred, average=avg),
                             Baseline=np.bincount(yte).max() / len(yte), _pred=pred))
    if kind == 'reg':  # mean-baseline row for context
        b = np.full(len(yte), ytr.mean()); rows.append(dict(Task=name, Kind=kind, Algorithm='(baseline: mean)', Primary='R²', Score=r2_score(yte, b), CV=0.0, RMSE=np.sqrt(mean_squared_error(yte, b)), MAE=mean_absolute_error(yte, b), MAPE=np.mean(np.abs((yte - b) / yte)) * 100, Baseline=0.0))
    STORE[name] = dict(models=fitted, split=(Xtr, Xte, ytr, yte), kind=kind, rows=rows)
    return rows

# ---- task registry: name → (design matrix, target, kind) ----
STORE, TASKS = {}, {}
TASKS['1 Admission'] = (X_app, m.Admitted, 'binary'); TASKS['1b Referral'] = (X_app, m.Referred, 'binary'); TASKS['4 High-risk patient'] = (X_app, m.HighRisk, 'binary')
bs = bf[bf.Rating.notna()]; TASKS['2 Satisfaction (L/M/H)'] = (make_X(bs, ['Diagnosis', 'Severity', 'BillStatus', 'PayMethod', 'Provider', 'ClaimStatus'], ['Amount', 'pay_ratio', 'HasClaim', 'claim_ratio']), bs.SatisfactionClass.map({'Low': 0, 'Medium': 1, 'High': 2}), 'multi')
cl = bf[bf.HasClaim == 1]; TASKS['3 Claim approval'] = (make_X(cl, ['Provider', 'Diagnosis', 'Severity', 'BillStatus', 'PayMethod'], ['Amount', 'ClaimAmount', 'claim_ratio', 'pay_ratio']), (cl.ClaimStatus == 'Approved').astype(int), 'binary')
TASKS['5 Revenue (billing amount)'] = (make_X(bf, ['Diagnosis', 'Severity'], []), bf.Amount, 'reg')
ad = m[m.Admitted == 1]; TASKS['6 Length of stay'] = (X_app.loc[ad.index].loc[:, lambda d: d.std() > 0], ad.LengthOfStayDays, 'reg')
ROWS = []
for name, (X, y, kind) in TASKS.items(): ROWS += run_task(name, X, y, kind); print(f'done: {name:28s} n={len(X):>6,}')
R = pd.DataFrame(ROWS)

In [ ]:
# ---- results: every task vs its naive baseline ----
show = R.drop(columns=['_pred']).round(3)
for name, d in show.groupby('Task', sort=False):
    base = d.Baseline.iloc[0]; kind = d.Kind.iloc[0]
    print(f'\n=== {name}  (n={len(TASKS[name][0]):,};', f'majority-class accuracy {base:.3f})' if kind != 'reg' else 'baseline R² = 0)')
    display(d.drop(columns=['Task', 'Kind', 'Baseline']).dropna(axis=1, how='all').set_index('Algorithm'))

In [ ]:
# ---- confusion matrices (best model per classification task, hold-out set) + ROC for binary tasks ----
cls_tasks = [t for t in STORE if STORE[t]['kind'] != 'reg']
fig, axes = plt.subplots(1, len(cls_tasks), figsize=(4.2 * len(cls_tasks), 4))
for ax, t in zip(axes, cls_tasks):
    d = R[(R.Task == t)]; best = d.loc[d.Score.idxmax()]; yte = STORE[t]['split'][3]
    ConfusionMatrixDisplay(confusion_matrix(yte, best._pred)).plot(ax=ax, colorbar=False, cmap='Blues'); ax.set_title(f'{t}\n{best.Algorithm}', fontsize=9)
plt.tight_layout(); plt.savefig(CFG['OUT'] + 'confusion_matrices.png', dpi=130); plt.show()

In [ ]:
# ---- risk-tier calibration for the high-risk model: should rise from Low → High if the model works ----
d = R[R.Task == '4 High-risk patient']; b4 = d.loc[d.Score.idxmax(), 'Algorithm']; Xtr, Xte, ytr, yte = STORE['4 High-risk patient']['split']
p = STORE['4 High-risk patient']['models'][b4].predict_proba(Xte)[:, 1]; tier = pd.qcut(pd.Series(p).rank(method='first'), [0, .6, .85, 1], labels=['Low', 'Medium', 'High'])
display(pd.DataFrame({'tier': tier.values, 'observed_escalation': yte}).groupby('tier').observed_escalation.agg(['count', 'mean']).round(3))
display(Markdown('**Interventions:** *High* – same-week clinician review, care-coordinator assignment, pre-alert receiving ward/referral site, medication reconciliation · *Medium* – nurse phone follow-up at 7 days, chronic-disease education, lab review within 48 h · *Low* – routine pathway, re-score next visit.'))

### MLflow tracking (and optional Unity Catalog registry with a promotion gate)

In [ ]:
def _log_model(mdl, Xs):
    kw = dict(signature=infer_signature(Xs, mdl.predict(Xs)), input_example=Xs.head(3), serialization_format='cloudpickle')   # cloudpickle avoids an optional-package dependency on newer MLflow
    try: return mlflow.sklearn.log_model(mdl, name='model', **kw)
    except TypeError: return mlflow.sklearn.log_model(mdl, artifact_path='model', **kw)

def passes_bar(row): return (row.CV >= CFG['DEPLOY_R2'] and row.Score > CFG['DEPLOY_R2']) if row.Kind == 'reg' else row.CV >= CFG['DEPLOY_AUC']

def log_to_mlflow():
    """One parent run, one nested run per task: all algorithm metrics, best model, and (optionally) a gated UC registration."""
    out = []
    with mlflow.start_run(run_name='HealthConnect pipeline') as parent:
        mlflow.log_params(dict(appointments=len(m), bills=len(bf), seed=RS, deploy_auc=CFG['DEPLOY_AUC'], deploy_r2=CFG['DEPLOY_R2']))
        for task, (X, y, kind) in TASKS.items():
            d = R[(R.Task == task) & (R.Algorithm != '(baseline: mean)')]; b = d.loc[d.Score.idxmax()]; ok = bool(passes_bar(b))
            with mlflow.start_run(run_name=task, nested=True) as run:
                mlflow.set_tags(dict(task=task, kind=kind, best_algorithm=b.Algorithm, passes_deployment_bar=str(ok)))
                mlflow.log_param('rows', len(X))
                for _, r in d.iterrows():
                    k = re.sub(r'[^0-9A-Za-z]+', '_', r.Algorithm); mlflow.log_metrics({f'{k}_score': float(r.Score), f'{k}_cv': float(r.CV)})
                mlflow.log_metrics(dict(best_score=float(b.Score), best_cv=float(b.CV), baseline=float(b.Baseline)))
                mdl = STORE[task]['models'][b.Algorithm]; info = _log_model(mdl, STORE[task]['split'][0].head(200))
                status = 'not registered'
                if REGISTER and CATALOG:
                    name = f'{CATALOG}.hc_' + re.sub(r'[^0-9a-z]+', '_', task.lower()).strip('_'); mv = mlflow.register_model(info.model_uri, name)
                    MlflowClient().set_registered_model_alias(name, 'champion' if ok else 'shadow', mv.version); status = f"{name} v{mv.version} → {'champion' if ok else 'shadow'}"
                out.append(dict(Task=task, Best=b.Algorithm, Score=round(float(b.Score), 3), CV=round(float(b.CV), 3), PassesBar=ok, Registry=status, RunId=run.info.run_id))
        for f in glob.glob(CFG['OUT'] + 'figures/*.png'): mlflow.log_artifact(f, 'figures')
    return pd.DataFrame(out)
MLRUNS = log_to_mlflow(); display(MLRUNS)

## 4 · Segmentation and capacity

In [ ]:
def segment_patients(m):
    ins = m.groupby('InsuranceID').apply(lambda d: pd.Series(dict(ins_paid_rate=(d.BillStatus == 'Paid').mean())))
    pa = m.groupby('PatientID').agg(Age=('Age', 'first'), Chronic=('ChronicCondition', lambda s: int(s.iloc[0] != 'None')), Visits=('AppointmentID', 'count'), BillTotal=('Amount', 'sum'), BillMean=('Amount', 'mean'), InsuranceID=('InsuranceID', 'first'))
    pa = pa.join(ins, on='InsuranceID').join(pd.crosstab(m.PatientID, m.Diagnosis, normalize='index').add_prefix('dx_')); feats = [c for c in pa.columns if c != 'InsuranceID']; Z = StandardScaler().fit_transform(pa[feats])
    sil = {k: silhouette_score(Z, KMeans(k, n_init=10, random_state=RS).fit_predict(Z)) for k in range(2, 9)}; k = max(sil, key=sil.get); pa['Segment'] = KMeans(k, n_init=20, random_state=RS).fit_predict(Z)
    prof = pa.groupby('Segment').agg(Patients=('Age', 'size'), Age=('Age', 'mean'), Visits=('Visits', 'mean'), BillMean=('BillMean', 'mean'), BillTotal=('BillTotal', 'mean'), **{c: (c, 'mean') for c in pa if c.startswith('dx_')}).round(2)
    return pa, prof, k, sil[k]
pa, prof, k, s = segment_patients(m); display(Markdown(f'**Patient segments: k = {k}, silhouette {s:.3f}.** Five are diagnosis-defined (most patients have one visit, so diagnosis mix is one-hot); the group with ~2 visits and ~2× billing is the care-coordination target.')); display(prof)

def segment_facilities(T, m):
    st = T['staff'].groupby('FacilityID').size().rename('Staff'); occ = (m[m.Admitted == 1].groupby('FacilityID').LengthOfStayDays.sum() / (m.groupby('FacilityID').facility_beds.first() * 365)).rename('Est_bed_occupancy')
    f = m.groupby('FacilityID').agg(Patient_volume=('AppointmentID', 'count'), Revenue=('Amount', 'sum'), Avg_rating=('Rating', 'mean'), Admission_rate=('Admitted', 'mean')).join(st).join(occ).fillna(0)
    Z = StandardScaler().fit_transform(f); sil = {k: silhouette_score(Z, KMeans(k, n_init=20, random_state=RS).fit_predict(Z)) for k in range(2, 6)}; k = max(sil, key=sil.get); f['Cluster'] = KMeans(k, n_init=50, random_state=RS).fit_predict(Z)
    return f, k, sil[k]
fac, kf, sf = segment_facilities(T, m); display(Markdown(f'**Facility segments: k = {kf}, silhouette {sf:.3f}** (20 facilities; weak structure → use for review priorities, not budgets).')); display(fac.groupby('Cluster').agg(Facilities=('Staff', 'size'), Volume=('Patient_volume', 'mean'), Revenue=('Revenue', 'mean'), Staff=('Staff', 'mean'), Rating=('Avg_rating', 'mean'), Est_occupancy=('Est_bed_occupancy', 'mean')).round(2))

def capacity_analysis(T, m):
    """Which departments need staff / which facilities are crowded? Uses the department→facility map (appointment FacilityID is unreliable)."""
    S = T['staff']; d = T['dept'][['DepartmentID', 'FacilityID', 'DepartmentName']].set_index('DepartmentID')
    d['clinical_staff'] = S[S.Role.isin(['Doctor', 'Nurse'])].groupby('DepartmentID').size(); d['beds'] = T['beds'].merge(T['rooms'][['RoomID', 'DepartmentID']], on='RoomID').groupby('DepartmentID').size(); d['appointments'] = m.groupby('DepartmentID').size()
    d = d.fillna(0); d['appts_per_staff'] = d.appointments / d.clinical_staff.replace(0, np.nan); d['appts_per_bed'] = d.appointments / d.beds.replace(0, np.nan)
    d['pressure'] = sum((d[c] - d[c].mean()) / d[c].std() for c in ['appts_per_staff', 'appts_per_bed']); fo = d.groupby('FacilityID').agg(appointments=('appointments', 'sum'), beds=('beds', 'sum')); fo['appts_per_bed'] = fo.appointments / fo.beds
    return d.sort_values('pressure', ascending=False), fo.sort_values('appts_per_bed', ascending=False)
dep, fo = capacity_analysis(T, m); display(Markdown('**Top-5 departments by staffing pressure** and **most crowded facilities** (appointments per bed):')); display(dep.head(5).round(2)); display(fo.head(4).round(2))
dep.to_csv(CFG['OUT'] + 'department_pressure.csv'); fo.to_csv(CFG['OUT'] + 'facility_crowding.csv')

## 5 · Anomaly detection
Rule-based and Isolation-Forest detectors; every flag is a **review lead**, not a finding of wrongdoing.

In [ ]:
def detect_anomalies(T, m, bf, patients):
    out, leads = {}, []
    def add(kind, ids, why): leads.extend({'Detector': kind, 'Entity': i, 'Reason': why} for i in ids)
    # E1/E2b unusually high bills, claim/payment outliers (Isolation Forest on all 60,000 bills)
    F = pd.DataFrame({'Amount': bf.Amount, 'pay_ratio': bf.pay_ratio, 'claim_ratio': bf.claim_ratio, **{c: bf[c].astype('category').cat.codes for c in ['Severity', 'PayMethod', 'Provider', 'BillStatus']}})
    sc = -IsolationForest(n_estimators=200, contamination=.005, random_state=RS).fit(F).score_samples(F); top = bf.assign(score=sc).nlargest(100, 'score'); add('High billing / claim outlier', top.BillingID, 'Isolation Forest on amount, payment & claim ratios'); out['High billing / claim outliers'] = len(top)
    # E2c status / payment / claim conflicts
    c = dict(paid_underpaid=((bf.BillStatus == 'Paid') & (bf.pay_ratio < 1)).sum(), pending_fully_paid=((bf.BillStatus == 'Pending') & (bf.pay_ratio >= 1)).sum(), insurance_payment_no_claim=((bf.PayMethod == 'Insurance') & (bf.HasClaim == 0)).sum())
    out['Status/payment/claim conflicts (bills)'] = int(((bf.BillStatus == 'Paid') & (bf.pay_ratio < 1) | (bf.BillStatus == 'Pending') & (bf.pay_ratio >= 1) | (bf.PayMethod == 'Insurance') & (bf.HasClaim == 0)).sum())
    # E3 abnormal labs: patients with ≥2 abnormal results
    ab = m[m.has_lab == 1].groupby('PatientID').lab_abn.sum(); ab = ab[ab >= 2]; add('Repeat abnormal labs', ab.index, '≥2 abnormal results'); out['Patients with ≥2 abnormal labs'] = len(ab)
    # E4 duplicate patients (name + DOB + gender + village; phone digits are truncated so excluded)
    dup = patients[patients.duplicated(['FirstName', 'LastName', 'DateOfBirth', 'Gender', 'VillageID'], keep=False)]; add('Duplicate patient', dup.PatientID, 'same name, DOB, gender, village'); out['Probable duplicate patient records'] = len(dup)
    # E5 prescribing: adult cardiometabolic drugs in children; doctors with unusual prescribing rate
    ch = m[(m.Age < 12) & m.Medication.isin(['Metformin', 'Amlodipine'])]; add('Age-inappropriate prescription', ch.AppointmentID, 'Metformin/Amlodipine under 12'); out['Age-inappropriate prescriptions'] = len(ch)
    dr = m.groupby('DoctorID').agg(n=('AppointmentID', 'count'), rx=('has_rx', 'sum')).query('n >= 8'); p0 = m.has_rx.mean(); z = (dr.rx / dr.n - p0) / np.sqrt(p0 * (1 - p0) / dr.n); out['Doctors with unusual prescribing rate'] = int((z.abs() > 2.5).sum())
    # E6 facility-month volume, E7 revenue months
    fv = m.groupby(['FacilityID', 'YM']).size().unstack(fill_value=0); mad = fv.sub(fv.median(1), axis=0).abs().median(1) * 1.4826; rz = fv.sub(fv.median(1), axis=0).div(mad.replace(0, np.nan), axis=0); out['Unusual facility-months (volume)'] = int((rz.abs() > 3.5).sum().sum())
    rv = T['revenue'].assign(rpa=lambda d: d.Revenue / d.Appointments); zz = (rv.rpa - rv.rpa.median()) / (1.4826 * (rv.rpa - rv.rpa.median()).abs().median()); zr = (rv.Revenue - rv.Revenue.median()) / (1.4826 * (rv.Revenue - rv.Revenue.median()).abs().median())
    bad = rv[(zz.abs() > 2.5) | (zr.abs() > 2.5)].Month.tolist(); out[f'Unusual revenue months {bad}'] = len(bad)
    return pd.Series(out, name='flagged').to_frame(), pd.DataFrame(leads), c
an, leads, conf = detect_anomalies(T, m, bf, patients); display(an); print('Conflict breakdown:', conf); leads.to_csv(CFG['OUT'] + 'anomaly_review_list.csv', index=False)

**Value:** ranked review lists focus audit effort on a few hundred cases instead of random sampling; duplicate-record and reconciliation findings are *data-quality* fixes at source (HIS); prescribing and billing flags reduce waste and fraud risk. Thresholds should be tuned with the audit team.
## 6 · Explainable AI (SHAP · permutation · LIME · bootstrap uncertainty)

In [ ]:
import shap
from lime.lime_tabular import LimeTabularExplainer
def explain_model(task='1 Admission'):
    d = R[(R.Task == task) & R.Algorithm.isin(['Random Forest', 'XGBoost', 'LightGBM'])]; name = d.loc[d.Score.idxmax(), 'Algorithm']; mdl = STORE[task]['models'][name]; Xtr, Xte, ytr, yte = STORE[task]['split']
    sv = shap.TreeExplainer(mdl).shap_values(Xte); sv = sv[1] if isinstance(sv, list) else (sv[:, :, 1] if np.ndim(sv) == 3 else sv)
    imp = pd.Series(np.abs(sv).mean(0), index=Xte.columns).sort_values(ascending=False)
    plt.figure(); shap.summary_plot(sv, Xte, max_display=12, show=False); plt.title(f'SHAP – {task} – {name}'); plt.tight_layout(); plt.savefig(CFG['OUT'] + 'shap_beeswarm.png', dpi=130); plt.show()
    pi = pd.Series(permutation_importance(mdl, Xte, yte, scoring='roc_auc', n_repeats=8, random_state=RS).importances_mean, index=Xte.columns).sort_values(ascending=False)
    pp = mdl.predict_proba(Xte)[:, 1]; rng = np.random.default_rng(RS); aucs = [roc_auc_score(yte[i], pp[i]) for i in (rng.integers(0, len(yte), len(yte)) for _ in range(300)) if len(set(yte[i])) > 1]; ci = np.percentile(aucs, [2.5, 97.5])
    lx = LimeTabularExplainer(Xtr.values, feature_names=list(Xtr.columns), class_names=['No', 'Yes'], mode='classification', random_state=RS)
    fig, ax = plt.subplots(1, 2, figsize=(13, 3.6))
    for a, i, t in zip(ax, [int(pp.argmax()), int(pp.argmin())], ['Highest predicted risk', 'Lowest predicted risk']):
        e = lx.explain_instance(Xte.values[i], lambda z: mdl.predict_proba(pd.DataFrame(z, columns=Xtr.columns)), num_features=8, num_samples=1500).as_list()
        a.barh([x[0][:34] for x in e][::-1], [x[1] for x in e][::-1], color=['C3' if x[1] > 0 else 'C0' for x in e][::-1]); a.set_title(f'LIME – {t} (p={pp[i]:.2f}, actual={int(yte[i])})', fontsize=9)
    plt.tight_layout(); plt.savefig(CFG['OUT'] + 'lime_local.png', dpi=130); plt.show()
    return name, imp, pi, ci
xname, imp, pi, ci = explain_model()
print(f'Model explained: {xname} | top SHAP: {list(imp.index[:5])} | top permutation: {list(pi.index[:3])}')
display(Markdown(f'**Bootstrap 95% AUC interval: {ci[0]:.2f}–{ci[1]:.2f}.** ' + ('It includes 0.50, so attributions describe noise-fitting and must **not** be read as clinical causes.' if ci[0] <= .52 else 'It excludes chance, so attributions reflect a real but modest signal.')))

**Why transparency matters:** clinicians can agree or override a flag with evidence; global plots expose unsafe dependencies (e.g. reliance on scheme/facility codes); subgroup audits need explanations; stored explanations form the audit trail; confidence intervals prevent over-reliance on weak models.
## 7 · Model comparison and deployment decision

In [ ]:
def compare_models(R):
    best = R[R.Algorithm != '(baseline: mean)'].loc[lambda d: d.groupby('Task').Score.idxmax()].copy()
    best['Beats baseline?'] = np.where(best.Kind == 'reg', best.Score > CFG['DEPLOY_R2'], best.CV >= CFG['DEPLOY_AUC'] if True else False)
    best['Strengths'] = best.Algorithm.map({'Logistic Regression': 'transparent, stable', 'Decision Tree': 'readable rules', 'Random Forest': 'robust to noise', 'XGBoost': 'strong on tabular data, SHAP', 'LightGBM': 'fast, efficient', 'Linear Regression': 'interpretable baseline', 'Gradient Boosting': 'smooth additive'})
    best['Recommended use'] = best.Task.map({'1 Admission': 'bed-demand early warning', '1b Referral': 'referral-capacity planning', '2 Satisfaction (L/M/H)': 'service-recovery outreach', '3 Claim approval': 'claims-desk prioritisation', '4 High-risk patient': 'care-coordination triage', '5 Revenue (billing amount)': 'aggregate budgeting only', '6 Length of stay': 'bed management via empirical distribution'})
    return best[['Task', 'Algorithm', 'Primary', 'Score', 'CV', 'Beats baseline?', 'Strengths', 'Recommended use']].round(3)
CMP = compare_models(R); display(CMP); CMP.to_csv(CFG['OUT'] + 'model_comparison.csv', index=False); R.drop(columns=['_pred']).to_csv(CFG['OUT'] + 'all_model_results.csv', index=False)

def decide(R):
    c = R[R.Kind == 'binary'].sort_values('CV', ascending=False).iloc[0]
    if c.CV >= CFG['DEPLOY_AUC']: return f"DEPLOY (shadow mode first): {c.Algorithm} for {c.Task}, CV AUC {c.CV:.3f}."
    return (f"NO MODEL CLEARS THE BAR (CV AUC ≥ {CFG['DEPLOY_AUC']}, R² > {CFG['DEPLOY_R2']}). Strongest candidate: {c.Algorithm} for '{c.Task}' (CV AUC {c.CV:.3f}) – indistinguishable from chance. "
            "Do not use any model for individual patient decisions; re-run on the full 60,000-appointment table and pilot in shadow mode. Anomaly, segmentation and capacity outputs are usable now.")
display(Markdown('### Decision\n' + decide(R)))

## 8 · Deployment architecture on Databricks (conceptual)
| Layer | Databricks component | In this project |
|---|---|---|
| Landing | Unity Catalog **Volumes** | raw files in `raw/`, daily CSV drops in `daily/` (optionally ingested with **Auto Loader**) |
| Bronze / Silver / Gold | **Delta tables** | `hc_appointment_clean`, `hc_billing_clean` (silver); model, anomaly, segment, capacity and drift tables (gold) |
| Training | This notebook as a **Workflow** task | scheduled retrain with identical preprocessing |
| Tracking and registry | **MLflow** + **Unity Catalog models** | every run logged; a model is aliased `champion` only if it clears the bar, else `shadow` |
| Serving | **Model Serving** endpoint (after a champion exists) | real-time scoring from the HIS; batch scoring for the nightly list |
| Dashboards | **AI/BI dashboards** on the gold tables | SQL in section 12 |
| Monitoring | Delta drift table + Lakehouse Monitoring | PSI per feature, model-health metrics, alerts |
| Governance | Unity Catalog permissions, audit logs | direct identifiers removed in silver; role-based access |

In [ ]:
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
fig, ax = plt.subplots(figsize=(13, 4.6)); ax.set_xlim(0, 13); ax.set_ylim(0, 4.6); ax.axis('off')
def box(x, y, w, t, c): ax.add_patch(FancyBboxPatch((x, y), w, 1.15, boxstyle='round,pad=0.04', fc=c, ec='#334')); ax.text(x + w / 2, y + .58, t, ha='center', va='center', fontsize=8)
steps = [('HIS / EMR\nappointments · labs · billing\npayments · claims · beds', '#dbeafe'), ('Ingestion + quality gates\nHL7 FHIR / REST / ETL', '#e0e7ff'), ('Feature pipeline\n(this notebook as a\nversioned pipeline)', '#ede9fe'), ('Model registry\nvalidation gates: AUC,\nfairness, clinical sign-off', '#fce7f3'), ('Scoring API\nreal-time + nightly batch\nSHAP reasons per score', '#dcfce7'), ('HIS + dashboards\nrisk flag · bed forecast\nanomaly queues', '#ffedd5')]
for i, (t, c) in enumerate(steps):
    x = .1 + i * 2.15; box(x, 2.7, 1.95, t, c)
    if i: ax.add_patch(FancyArrowPatch((x - .2, 3.28), (x, 3.28), arrowstyle='-|>', mutation_scale=12, color='#334'))
for x, t, c in [(.1, 'Monitoring: data/prediction drift,\nrealised-outcome AUC, subgroup fairness', '#fee2e2'), (4.6, 'Governance: RBAC, encryption, audit log,\nconsent, human-in-the-loop sign-off', '#f1f5f9'), (9.1, 'Retraining: quarterly or AUC drop >5 pts;\nchampion/challenger shadow; rollback', '#ecfccb')]: box(x, .5, 3.7, t, c)
plt.title('HealthConnect Rwanda – ML deployment architecture'); plt.savefig(CFG['OUT'] + 'deployment_architecture.png', dpi=140, bbox_inches='tight'); plt.show()

## 9 · Ethical and responsible AI
| Theme | Risk here | Control |
|---|---|---|
| Privacy & confidentiality | Names, phones and emergency contacts in source data | Identifiers dropped from exported datasets; pseudonymous IDs; purpose limitation under data-protection law |
| Data security | Features and scores reveal health status | Encryption, role-based access, audit logging |
| Bias & fairness | Scheme, age, gender, residence and facility can proxy socio-economic status | Subgroup metrics as a release gate; scores *prioritise* support, never deny care |
| Explainability | Black-box scores unsafe for clinical use | SHAP/LIME per prediction, interpretable baselines, model card |
| Human oversight | Automation bias | Decision support only; clinician override; escalation path |
| Governance | Drift, unclear ownership | AI governance committee, approval workflow, audits, retirement criteria |
| Validity | Weak signal, truncated appointments, contradictory fields | Baselines + intervals, shadow mode, no deployment below the bar |

In [ ]:
# ---- export a machine-readable summary ----
SUMMARY = dict(appointments_linked=len(m), billing_rows=len(bf), tasks={t: dict(best=CMP.set_index('Task').loc[t, 'Algorithm'], score=float(CMP.set_index('Task').loc[t, 'Score']), cv=float(CMP.set_index('Task').loc[t, 'CV'])) for t in CMP.Task},
               anomalies=an.flagged.to_dict(), patient_segments=int(k), facility_clusters=int(kf), explained_model=xname, auc_ci=[float(ci[0]), float(ci[1])], decision=decide(R))
json.dump(SUMMARY, open(CFG['OUT'] + 'summary.json', 'w'), indent=1, default=str); print('Saved to', CFG['OUT'], '→', sorted(os.listdir(CFG['OUT'])))

## 10 · Gold layer: tables for dashboards and downstream use

In [ ]:
gold = {'hc_model_results': R.drop(columns=['_pred']), 'hc_model_comparison': CMP, 'hc_mlflow_runs': MLRUNS, 'hc_anomaly_summary': an.reset_index().rename(columns={'index': 'detector'}),
        'hc_anomaly_leads': leads, 'hc_patient_segments': pa.reset_index(), 'hc_facility_segments': fac.reset_index(), 'hc_department_pressure': dep.reset_index(), 'hc_facility_crowding': fo.reset_index()}
for name, df in gold.items(): write_table(df, name)
display(pd.Series(TABLES, name='location').to_frame())

## 11 · Daily operations: ingest → validate → drift → retrain → gate → score
Each day a CSV drop lands in the `daily_dir` Volume (template columns: `Date, Age, Gender, ChronicCondition, VisitType, Diagnosis, Severity, BMI, PrevAppointments, PrevAdmissions, Admitted, Referred, Amount, LengthOfStayDays, Rating`; required: Age, VisitType, Diagnosis, Severity, Admitted).
New files are validated, compared with the baseline (population stability index), added to the training data, and the light operational models (logistic regression) are retrained and logged to MLflow. A model is flagged `champion` only if its cross-validated AUC reaches **0.70**; otherwise it runs in **shadow mode** and predictions are labelled accordingly.

In [ ]:
CATS = dict(Gender=['Female', 'Male'], ChronicCondition=['None', 'Asthma', 'Diabetes', 'Hypertension'], VisitType=['Consultation', 'Emergency', 'Follow-up'],
            Diagnosis=['Diabetes', 'Gastritis', 'Hypertension', 'Malaria', 'Upper Respiratory Infection'], Severity=['Mild', 'Moderate', 'Severe'])
ALIAS = dict(Age=['age'], Gender=['gender', 'sex'], ChronicCondition=['chroniccondition', 'chronic'], VisitType=['visittype', 'visit'], Diagnosis=['diagnosis'], Severity=['severity'],
             BMI=['bmi', 'bmiclean'], PrevAppointments=['prevappointments', 'nprevappointments', 'previousappointments'], PrevAdmissions=['prevadmissions', 'nprevadmissions', 'previousadmissions'],
             Date=['date', 'appointmentdate'], Month=['month'], Admitted=['admitted', 'admission'], Referred=['referred', 'referral'], Amount=['amount', 'billingamount'],
             LengthOfStayDays=['lengthofstaydays', 'lengthofstay', 'los'], Rating=['rating'])
REQUIRED = ['Age', 'VisitType', 'Diagnosis', 'Severity', 'Admitted']; RAINY = [3, 4, 5, 9, 10, 11, 12]
def _match(col, v):
    v = str(v).strip().lower()
    if col == 'ChronicCondition' and v in ('', 'none', 'n/a', 'na', 'nan', 'null', 'no'): return 'None'
    if col == 'Gender' and v in ('f', 'm'): return 'Female' if v == 'f' else 'Male'
    for x in CATS[col]:
        if x.lower() == v: return x
    return 'Upper Respiratory Infection' if col == 'Diagnosis' and 'respir' in v else None
_bool = lambda v: 1.0 if str(v).strip().lower() in ('1', '1.0', 'true', 'yes', 'y', 'admitted', 'referred') else 0.0 if str(v).strip().lower() in ('0', '0.0', 'false', 'no', 'n') else np.nan

def validate_daily(raw):
    """Map aliases, enforce required columns/ranges/categories; return (clean frame, report)."""
    norm = {re.sub(r'[^a-z0-9]', '', c.lower()): c for c in raw.columns}; cols = {k: next(norm[a] for a in al if a in norm) for k, al in ALIAS.items() if any(a in norm for a in al)}
    miss = [k for k in REQUIRED if k not in cols]
    if miss: raise ValueError(f'Missing required column(s): {miss}')
    d = pd.DataFrame({k: raw[v] for k, v in cols.items()}); rep = dict(read=len(d), rejected={}, filled={})
    def drop(mask, why):
        nonlocal d
        if mask.sum(): rep['rejected'][why] = int(mask.sum())
        d = d[~mask].copy()
    d['Age'] = pd.to_numeric(d.Age, errors='coerce'); drop(~d.Age.between(0, 120), 'invalid age')
    for c in ['VisitType', 'Diagnosis', 'Severity']: d[c] = d[c].map(lambda v: _match(c, v)); drop(d[c].isna(), f'unknown {c}')
    d['Admitted'] = d.Admitted.map(_bool); drop(d.Admitted.isna(), 'unreadable Admitted')
    for c, default in [('Gender', 'Female'), ('ChronicCondition', 'None')]:
        d[c] = d[c].map(lambda v: _match(c, v)) if c in d else None; n = int(d[c].isna().sum()); d[c] = d[c].fillna(default)
        if n: rep['filled'][c] = n
    d['Referred'] = d.Referred.map(_bool) if 'Referred' in d else np.nan
    for c, default in [('BMI', 27.0), ('PrevAppointments', 0.0), ('PrevAdmissions', 0.0)]:
        d[c] = pd.to_numeric(d[c], errors='coerce') if c in d else np.nan; n = int(d[c].isna().sum()); d[c] = d[c].fillna(default)
        if n: rep['filled'][c] = n
    mo = pd.to_numeric(d.Month, errors='coerce') if 'Month' in d else pd.Series(np.nan, index=d.index)
    if 'Date' in d: mo = mo.fillna(pd.to_datetime(d.Date, errors='coerce').dt.month)
    n = int(mo.isna().sum()); d['Month'] = mo.fillna(6).astype(int)
    if n: rep['filled']['Month'] = n
    d['HighRisk'] = np.where(d.Referred.notna(), ((d.Admitted == 1) | (d.Referred == 1)).astype(float), np.nan)
    rep['accepted'] = len(d); return d[['Age', 'Gender', 'ChronicCondition', 'VisitType', 'Diagnosis', 'Severity', 'BMI', 'PrevAppointments', 'PrevAdmissions', 'Month', 'Admitted', 'Referred', 'HighRisk']].reset_index(drop=True), rep

def light_from_master(m):
    d = pd.DataFrame(dict(Age=m.Age, Gender=m.Gender, ChronicCondition=m.ChronicCondition, VisitType=m.VisitType, Diagnosis=m.Diagnosis, Severity=m.Severity, BMI=m.BMI_clean,
                          PrevAppointments=m.n_prev_appointments, PrevAdmissions=m.n_prev_admissions, Month=m.Month, Admitted=m.Admitted.astype(float), Referred=m.Referred.astype(float)))
    d['HighRisk'] = ((d.Admitted == 1) | (d.Referred == 1)).astype(float); return d.reset_index(drop=True)

def light_X(d):
    X = d[['Age', 'BMI', 'PrevAppointments', 'PrevAdmissions']].astype(float).reset_index(drop=True)
    for c, L in CATS.items(): X = X.join(pd.get_dummies(pd.Categorical(d[c].values, categories=L), prefix=c, dtype=float).iloc[:, 1:])
    X['Rainy'] = d.Month.isin(RAINY).astype(float).values; return X

def train_light(d, target):
    d = d.dropna(subset=[target]).reset_index(drop=True); X, y = light_X(d), d[target].astype(int)
    if y.sum() < 10 or (len(y) - y.sum()) < 10: return None
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=.2, stratify=y, random_state=RS); mdl = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, C=0.2))
    cv = cross_val_score(mdl, Xtr, ytr, cv=StratifiedKFold(5, shuffle=True, random_state=RS), scoring='roc_auc').mean(); mdl.fit(Xtr, ytr)
    auc_ = roc_auc_score(yte, mdl.predict_proba(Xte)[:, 1]); final = clone(mdl).fit(X, y); ps = np.sort(final.predict_proba(X)[:, 1])
    return dict(model=final, auc=auc_, cv=cv, base=float(y.mean()), n=len(y), t60=float(ps[int(len(ps) * .6)]), t85=float(ps[int(len(ps) * .85)]), Xs=X.head(200))

def psi(a, b, n):
    f = lambda v: np.maximum(np.bincount(v, minlength=n) / max(len(v), 1), .0005); p, q = f(a), f(b); return float(((q - p) * np.log(q / p)).sum())
def drift_report(base, new):
    age_bin = lambda s: np.digitize(s, [5, 15, 25, 45, 65]); bmi_bin = lambda s: np.digitize(s, [18.5, 25, 30])
    feats = [('Age', 6, age_bin(base.Age), age_bin(new.Age)), ('BMI band', 4, bmi_bin(base.BMI), bmi_bin(new.BMI))] + \
            [(c, len(L), base[c].map({v: i for i, v in enumerate(L)}).fillna(0).astype(int).values, new[c].map({v: i for i, v in enumerate(L)}).fillna(0).astype(int).values) for c, L in CATS.items()]
    r = pd.DataFrame([(n, psi(a, b, k)) for n, k, a, b in feats], columns=['feature', 'psi']); r['status'] = np.where(r.psi > .2, 'Shift', np.where(r.psi > .1, 'Watch', 'Stable'))
    r.loc[len(r)] = ['Admission rate (pp change)', float((new.Admitted.mean() - base.Admitted.mean()) * 100), 'Info']; return r

In [ ]:
PROC = Path(CFG['OUT']) / 'tables' / 'daily_processed_files.csv'

def daily_run(daily_dir=DAILY_DIR):
    files = sorted(glob.glob(f'{_local(daily_dir).rstrip("/")}/*.csv')); done = set(pd.read_csv(PROC).file) if PROC.exists() else set()
    todo = [f for f in files if os.path.basename(f) not in done]
    if not todo: print(f'No new daily files in {daily_dir} (found {len(files)}, already processed {len(done)}).'); return None
    frames, reps = [], []
    for f in todo:
        try: d, rep = validate_daily(pd.read_csv(f)); d['SourceFile'] = os.path.basename(f); frames.append(d); reps.append(dict(file=os.path.basename(f), **rep))
        except Exception as e: reps.append(dict(file=os.path.basename(f), error=str(e)))
    display(pd.DataFrame(reps))
    if not frames: return None
    new = pd.concat(frames, ignore_index=True); base = light_from_master(m); cols = [c for c in base.columns]
    drift = drift_report(base, new[cols]); drift['run_date'] = pd.Timestamp.today().strftime('%Y-%m-%d'); display(drift)
    train = pd.concat([base, new[cols]], ignore_index=True); health, preds = [], new.copy(); mode = {}
    with mlflow.start_run(run_name=f'daily-retrain {pd.Timestamp.today():%Y-%m-%d}'):
        mlflow.log_params(dict(new_rows=len(new), training_rows=len(train), files=len(todo)))
        for tgt in ['Admitted', 'Referred', 'HighRisk']:
            r = train_light(train, tgt)
            if r is None: continue
            ok = r['cv'] >= CFG['DEPLOY_AUC']; mode[tgt] = 'champion' if ok else 'shadow'
            with mlflow.start_run(run_name=f'light-{tgt}', nested=True):
                mlflow.set_tags(dict(target=tgt, status=mode[tgt])); mlflow.log_metrics(dict(holdout_auc=r['auc'], cv_auc=r['cv'], base_rate=r['base'], rows=r['n']))
                info = _log_model(r['model'], r['Xs'])
                if REGISTER and CATALOG:
                    nm = f'{CATALOG}.hc_light_{tgt.lower()}'; mv = mlflow.register_model(info.model_uri, nm); MlflowClient().set_registered_model_alias(nm, mode[tgt], mv.version)
            p = r['model'].predict_proba(light_X(new[cols]))[:, 1]; preds[f'p_{tgt}'] = p; preds[f'tier_{tgt}'] = np.where(p >= r['t85'], 'High', np.where(p >= r['t60'], 'Medium', 'Low'))
            health.append(dict(target=tgt, rows=r['n'], base_rate=round(r['base'], 3), holdout_auc=round(r['auc'], 3), cv_auc=round(r['cv'], 3), status=mode[tgt]))
        mlflow.log_dict(drift.to_dict(orient='records'), 'drift_report.json')
    preds['model_status'] = mode.get('Admitted', 'shadow')
    h = pd.DataFrame(health); display(h)
    write_table(drift, 'hc_daily_drift'); write_table(preds, 'hc_daily_predictions'); write_table(h, 'hc_daily_model_health')
    log_df = pd.DataFrame({'file': [os.path.basename(f) for f in todo], 'processed_at': pd.Timestamp.now().isoformat()})
    (pd.concat([pd.read_csv(PROC), log_df]) if PROC.exists() else log_df).to_csv(PROC, index=False)
    best = h.cv_auc.max() if len(h) else float('nan')
    print(f'Processed {len(todo)} file(s), {len(new)} rows. Best cross-validated AUC {best:.3f} → ' + ('CHAMPION candidate (still shadow-test and run a fairness audit before clinical use).' if best >= CFG['DEPLOY_AUC'] else 'SHADOW mode: below the 0.70 bar, predictions are for monitoring only.'))
    return dict(drift=drift, health=h, predictions=preds)

DAILY = daily_run()

## 12 · AI/BI dashboard queries (Databricks SQL)
Create an **AI/BI dashboard** on the gold tables; replace `main.healthconnect` with your `catalog_schema`.
```sql
-- Admission and referral rate by province
SELECT ProvinceName, ROUND(AVG(Admitted)*100,1) AS admission_pct, ROUND(AVG(Referred)*100,1) AS referral_pct, COUNT(*) AS appointments
FROM main.healthconnect.hc_appointment_clean GROUP BY ProvinceName ORDER BY admission_pct DESC;

-- Model scorecard against the deployment bar
SELECT Task, Algorithm, Primary, Score, CV, `Beats_baseline` AS clears_bar FROM main.healthconnect.hc_model_comparison;

-- Top departments by staffing pressure
SELECT DepartmentID, FacilityID, DepartmentName, appts_per_staff, appts_per_bed, pressure
FROM main.healthconnect.hc_department_pressure ORDER BY pressure DESC LIMIT 10;

-- Billing / payment conflicts
SELECT BillStatus, ROUND(AVG(pay_ratio)*100,1) AS avg_paid_pct, COUNT(*) AS bills FROM main.healthconnect.hc_billing_clean GROUP BY BillStatus;

-- Latest daily drift and model health
SELECT * FROM main.healthconnect.hc_daily_drift ORDER BY psi DESC;
SELECT * FROM main.healthconnect.hc_daily_model_health;
```
## 13 · Schedule as a daily Workflow
**Workflows → Create job → Notebook task** pointing at this notebook, cluster: a small job cluster on the ML runtime, schedule `0 0 2 * * ?` (daily 02:00), parameters = the widget names. Equivalent job JSON:
```json
{
  "name": "healthconnect-daily",
  "tasks": [{"task_key": "pipeline", "notebook_task": {"notebook_path": "/Workspace/Shared/HealthConnect_Databricks",
             "base_parameters": {"raw_dir": "/Volumes/main/healthconnect/raw", "out_dir": "/Volumes/main/healthconnect/outputs",
                                 "daily_dir": "/Volumes/main/healthconnect/daily", "catalog_schema": "main.healthconnect",
                                 "mlflow_experiment": "/Shared/healthconnect-ml", "register_models": "true"}},
             "new_cluster": {"spark_version": "15.4.x-cpu-ml-scala2.12", "node_type_id": "Standard_DS3_v2", "num_workers": 0,
                             "spark_conf": {"spark.databricks.cluster.profile": "singleNode", "spark.master": "local[*]"}, "custom_tags": {"ResourceClass": "SingleNode"}}}],
  "schedule": {"quartz_cron_expression": "0 0 2 * * ?", "timezone_id": "Africa/Kigali"}
}
```
For heavy daily volumes, split into two jobs: (1) the one-off or weekly full pipeline (sections 1 to 10) and (2) the daily task (section 11 only). Alert on `Shift` rows in `hc_daily_drift`.

## 14 · Project finalisation checklist

In [ ]:
exp = mlflow.get_experiment_by_name(EXPERIMENT)
checklist = pd.DataFrame([
    ('Raw data validated', 'OK', f'{len(T)} tables loaded from {RAW_DIR}'),
    ('Silver tables written', 'OK' if {'hc_appointment_clean', 'hc_billing_clean'} <= set(TABLES) else 'MISSING', ', '.join(t for t in TABLES if t in ('hc_appointment_clean', 'hc_billing_clean'))),
    ('Models trained and tracked in MLflow', 'OK' if len(MLRUNS) == len(TASKS) else 'CHECK', f'{len(MLRUNS)} task runs in experiment {EXPERIMENT}'),
    ('Models clearing the deployment bar', f"{int(MLRUNS.PassesBar.sum())} of {len(MLRUNS)}", 'bar: CV AUC ≥ 0.70, R² > 0.05'),
    ('Explainability (SHAP, LIME, permutation, bootstrap CI)', 'OK', f'explained {xname}; AUC 95% CI {ci[0]:.2f}–{ci[1]:.2f}'),
    ('Anomaly review lists', 'OK', f'{len(leads):,} leads in hc_anomaly_leads'),
    ('Gold tables for dashboards', 'OK' if 'hc_model_comparison' in TABLES else 'MISSING', f'{len(TABLES)} tables written'),
    ('Daily ingestion run', 'RAN' if DAILY else 'NO NEW FILES', DAILY_DIR),
    ('Deployment decision', 'SHADOW' if MLRUNS.PassesBar.sum() == 0 else 'REVIEW', decide(R)[:160] + '…')], columns=['Item', 'Status', 'Detail'])
display(checklist); checklist.to_csv(CFG['OUT'] + 'tables/finalisation_checklist.csv', index=False)
print('Experiment:', getattr(exp, 'experiment_id', 'n/a'), '| outputs:', OUT_DIR)